# PPR vs k-hop Subgraph Comparison

Empirically measures how classic restart α affects PPR subgraph composition,
and compares directly against k-hop subgraphs on the same seed pairs.

**Convention:** α here is the *classic PageRank restart probability* (high α = local, low α = global).
Internally `local_ppr.py` uses continuation = 1 − α, but all user-facing values in this notebook
follow the classic convention.

**Theoretical expected depth:** E[depth] = (1−α)/α
| α_restart | E[depth] |
|-----------|----------|
| 0.90      | 0.11     |
| 0.50      | 1.00     |
| 0.33      | 2.03     |
| 0.25      | 3.00     |
| 0.15      | 5.67     |

**Goal:** verify these theoretical values empirically and choose teleport_values for Learnable PPR.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
DATASET       = 'Cora'    # Planetoid dataset to test on
N_PAIRS       = 100       # number of seed pairs to sample
PPR_EPSILON   = 1e-3      # PPR approximation threshold
PPR_WINDOW    = 10        # conductance sweep window
SAVE_PATH     = 'ppr_khop_comparison.pkl'

# Classic restart alphas (high = local, like LCILP paper uses alpha=0.15)
CLASSIC_ALPHAS = [0.90, 0.50, 0.33, 0.25, 0.15]
# Internally used continuation alphas = 1 - CLASSIC_ALPHAS
# = [0.10, 0.50, 0.67, 0.75, 0.85]

K_HOPS = [1, 2, 3]


In [ ]:
import os, sys, pickle
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from torch_geometric.utils import k_hop_subgraph

from subgrapher.benchmark_ppr.ppr_extractor import StaticPPRExtractor
from subgrapher.utils.local_ppr import build_sparse_adj, approximate_ppr


In [ ]:
# Load from processed .pt file directly — bypasses PyG's aiohttp download.
# Works even when no internet is available.
for _root in ['data/Planetoid', 'data/planetoid']:
    _proc = os.path.join(_root, DATASET, 'processed', 'data.pt')
    if os.path.exists(_proc):
        _loaded = torch.load(_proc, weights_only=False)
        break
else:
    raise FileNotFoundError(
        f'{DATASET} processed data not found. '
        f'Run any other notebook that loads it first, or place '
        f'data/Planetoid/{DATASET}/processed/data.pt in the working directory.')

# Reconstruct Data — handles both old (collated, slices) and new (list) PyG formats
from torch_geometric.data import Data
if isinstance(_loaded, list) and hasattr(_loaded[0], 'edge_index'):
    data = _loaded[0]
elif isinstance(_loaded, tuple) and len(_loaded) == 2:
    _collated, _slices = _loaded
    data = Data(
        x=_collated.x,
        edge_index=_collated.edge_index,
        y=_collated.y if hasattr(_collated, 'y') else None,
    )
else:
    data = _loaded

if not hasattr(data, '_num_nodes') or data.num_nodes is None:
    data.num_nodes = int(data.x.size(0))

print(f'Loaded {DATASET}: {data.num_nodes} nodes, {data.edge_index.size(1)} edges')

# Build adjacency for BFS distance queries
adj_csr = build_sparse_adj(data.edge_index, data.num_nodes)

# Sample N_PAIRS connected pairs from edge_index
torch.manual_seed(42)
perm = torch.randperm(data.edge_index.size(1))[:N_PAIRS]
pairs = [(int(data.edge_index[0, i]), int(data.edge_index[1, i])) for i in perm]
print(f'Sampled {len(pairs)} seed pairs')


In [ ]:
# Build one PPR extractor per classic restart alpha.
# Internally passes continuation = 1 - classic_alpha to the push algorithm.
print('Building PPR extractors (one per alpha)...')
extractors = {}
for ca in CLASSIC_ALPHAS:
    cont = 1.0 - ca
    e_depth = (1 - ca) / ca
    print(f'  α_restart={ca:.2f} → α_cont={cont:.2f}  E[depth]={e_depth:.2f}')
    extractors[ca] = StaticPPRExtractor(
        data, alpha=cont, epsilon=PPR_EPSILON, window=PPR_WINDOW)


In [ ]:
def bfs_max_dist(adj, seed_nodes, node_set):
    """Max BFS distance from any seed to any node in node_set."""
    visited = {s: 0 for s in seed_nodes}
    queue = list(seed_nodes)
    while queue:
        node = queue.pop(0)
        for nb in adj[node].indices:
            if nb not in visited:
                visited[nb] = visited[node] + 1
                queue.append(nb)
    dists = [visited.get(n, 0) for n in node_set if n not in seed_nodes]
    return max(dists) if dists else 0


khop_sets  = {k: [] for k in K_HOPS}
ppr_sets   = {ca: [] for ca in CLASSIC_ALPHAS}
ppr_scores = {ca: [] for ca in CLASSIC_ALPHAS}

print(f'Extracting subgraphs for {len(pairs)} pairs...')
for i, (u, v) in enumerate(pairs):
    if i % 25 == 0:
        print(f'  [{i}/{len(pairs)}]')

    # k-hop subgraphs
    for k in K_HOPS:
        node_ids, _, _, _ = k_hop_subgraph(
            [u, v], k, data.edge_index,
            relabel_nodes=False, num_nodes=data.num_nodes)
        khop_sets[k].append(set(node_ids.tolist()))

    # PPR subgraphs
    for ca in CLASSIC_ALPHAS:
        cont = 1.0 - ca
        _, selected_nodes, _ = extractors[ca].extract_subgraph(u, v)
        ppr_sets[ca].append(set(selected_nodes.tolist()))
        # Raw PPR scores for visualization
        raw = approximate_ppr(adj_csr, {u, v}, alpha=cont, epsilon=PPR_EPSILON)
        ppr_scores[ca].append({int(n): float(raw[n]) for n in np.nonzero(raw)[0]})

print('Done.')


In [ ]:
# ── Statistics ──────────────────────────────────────────────────────────────
stats = {}
for ca in CLASSIC_ALPHAS:
    sizes = [len(s) for s in ppr_sets[ca]]
    max_depths = []
    for i, (u, v) in enumerate(pairs):
        d = bfs_max_dist(adj_csr, {u, v}, ppr_sets[ca][i])
        max_depths.append(d)
    stats[ca] = {
        'mean_size':       np.mean(sizes),
        'median_size':     np.median(sizes),
        'mean_max_depth':  np.mean(max_depths),
        'e_depth_theory':  (1 - ca) / ca,
    }

khop_stats = {}
for k in K_HOPS:
    sizes = [len(s) for s in khop_sets[k]]
    khop_stats[k] = {'mean_size': np.mean(sizes)}

print(f'{'α_restart':>10}  {'α_cont':>6}  {'E[depth]_theory':>16}  '
      f'{'E[depth]_actual':>16}  {'mean_nodes':>10}')
for ca in CLASSIC_ALPHAS:
    s = stats[ca]
    print(f'{ca:>10.2f}  {1-ca:>6.2f}  {s["e_depth_theory"]:>16.2f}  '
          f'{s["mean_max_depth"]:>16.2f}  {s["mean_size"]:>10.1f}')
print()
for k in K_HOPS:
    print(f'  k={k}-hop: mean_nodes={khop_stats[k]["mean_size"]:.1f}')


In [ ]:
# ── Plots ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1: Mean subgraph size vs alpha
ax = axes[0]
ppr_sizes = [stats[ca]['mean_size'] for ca in CLASSIC_ALPHAS]
ax.plot(CLASSIC_ALPHAS, ppr_sizes, 'bo-', label='PPR')
colors = ['green', 'orange', 'red']
for k, col in zip(K_HOPS, colors):
    ax.axhline(khop_stats[k]['mean_size'], color=col, linestyle='--',
               label=f'{k}-hop (mean={khop_stats[k]["mean_size"]:.0f})')
ax.set_xlabel('Classic restart α  (← global | local →)')
ax.set_ylabel('Mean subgraph size (nodes)')
ax.set_title('Subgraph Size vs α')
ax.invert_xaxis()
ax.legend(fontsize=8)

# 2: Actual max depth vs theoretical
ax = axes[1]
actual = [stats[ca]['mean_max_depth'] for ca in CLASSIC_ALPHAS]
theory = [(1 - ca) / ca for ca in CLASSIC_ALPHAS]
ax.plot(CLASSIC_ALPHAS, actual, 'ro-', label='Actual max depth (mean)')
ax.plot(CLASSIC_ALPHAS, theory, 'k--', label='Theory: (1−α)/α')
for k, col in zip(K_HOPS, colors):
    ax.axhline(k, color=col, linestyle=':', alpha=0.6, label=f'{k}-hop baseline')
ax.set_xlabel('Classic restart α  (← global | local →)')
ax.set_ylabel('Max hop depth from seed')
ax.set_title('Depth vs α: Actual vs Theory')
ax.invert_xaxis()
ax.legend(fontsize=8)

# 3: Jaccard overlap PPR vs k-hop
ax = axes[2]
for k, col in zip(K_HOPS, colors):
    overlaps = []
    for ca in CLASSIC_ALPHAS:
        ratios = []
        for i in range(len(pairs)):
            inter = ppr_sets[ca][i] & khop_sets[k][i]
            union = ppr_sets[ca][i] | khop_sets[k][i]
            ratios.append(len(inter) / len(union) if union else 0.0)
        overlaps.append(np.mean(ratios))
    ax.plot(CLASSIC_ALPHAS, overlaps, 'o-', color=col, label=f'{k}-hop')
ax.set_xlabel('Classic restart α  (← global | local →)')
ax.set_ylabel('Jaccard overlap with k-hop')
ax.set_title('PPR ∩ k-hop Overlap')
ax.invert_xaxis()
ax.legend(fontsize=8)

plt.suptitle(f'{DATASET}: PPR vs k-hop Subgraph Comparison', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('ppr_khop_stats.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved ppr_khop_stats.png')


In [ ]:
# ── Example visualization: one pair, 2-hop vs PPR α=0.33 ─────────────────
try:
    import networkx as nx

    pair_idx = 0
    u, v = pairs[pair_idx]

    # Union of all subgraphs for this pair → visualization graph
    all_vis_nodes = set([u, v])
    for k in K_HOPS:
        all_vis_nodes |= khop_sets[k][pair_idx]
    for ca in CLASSIC_ALPHAS:
        all_vis_nodes |= ppr_sets[ca][pair_idx]

    node_set = set(all_vis_nodes)
    ei = data.edge_index
    mask = ((ei[0].apply_(lambda x: x in node_set).bool()) &
            (ei[1].apply_(lambda x: x in node_set).bool()))
    # Safer alternative:
    src_np = ei[0].numpy()
    dst_np = ei[1].numpy()
    mask = np.array([src_np[i] in node_set and dst_np[i] in node_set
                     for i in range(len(src_np))])
    sub_ei = ei[:, torch.from_numpy(mask)]

    G_vis = nx.Graph()
    G_vis.add_nodes_from(sorted(all_vis_nodes))
    G_vis.add_edges_from(sub_ei.t().tolist())
    pos = nx.spring_layout(G_vis, seed=42)

    def draw_sub(ax, title, highlight):
        colors = ['red' if n in (u, v) else
                  'dodgerblue' if n in highlight else
                  'lightgray'
                  for n in G_vis.nodes()]
        nx.draw(G_vis, pos=pos, ax=ax, node_color=colors,
                node_size=60, with_labels=False,
                edge_color='lightgray', width=0.5)
        ax.set_title(title, fontsize=9)
        patches = [
            mpatches.Patch(color='red', label='Seed (u,v)'),
            mpatches.Patch(color='dodgerblue', label='In subgraph'),
            mpatches.Patch(color='lightgray', label='Excluded'),
        ]
        ax.legend(handles=patches, fontsize=7, loc='upper right')

    fig, axes = plt.subplots(1, len(CLASSIC_ALPHAS) + len(K_HOPS),
                              figsize=(4 * (len(CLASSIC_ALPHAS) + len(K_HOPS)), 4))
    for i, k in enumerate(K_HOPS):
        draw_sub(axes[i], f'{k}-hop  (n={len(khop_sets[k][pair_idx])})',
                 khop_sets[k][pair_idx])
    for j, ca in enumerate(CLASSIC_ALPHAS):
        e_d = (1 - ca) / ca
        draw_sub(axes[len(K_HOPS) + j],
                 f'PPR α={ca}  E[d]={e_d:.1f}  (n={len(ppr_sets[ca][pair_idx])})',
                 ppr_sets[ca][pair_idx])

    plt.suptitle(f'Pair {pair_idx}: u={u}, v={v}', fontsize=10)
    plt.tight_layout()
    plt.savefig('ppr_vs_khop_example.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved ppr_vs_khop_example.png')
except Exception as e:
    print(f'Visualization skipped (networkx not available?): {e}')


In [ ]:
# ── Save comparison data ─────────────────────────────────────────────────
result = {
    'dataset':       DATASET,
    'seed_pairs':    pairs,
    'classic_alphas': CLASSIC_ALPHAS,
    'k_hops':        K_HOPS,
    # Subgraph node sets (serialized as sorted lists)
    'khop':          {k:  [sorted(s) for s in khop_sets[k]]  for k in K_HOPS},
    'ppr':           {ca: [sorted(s) for s in ppr_sets[ca]]  for ca in CLASSIC_ALPHAS},
    # Raw PPR scores: {classic_alpha: [{node_id: score}, ...]}
    'ppr_scores':    ppr_scores,
    # Full graph topology (for downstream visualization)
    'edge_index':    data.edge_index.cpu(),
    'num_nodes':     data.num_nodes,
    # Pre-computed statistics
    'stats':         stats,
    'khop_stats':    khop_stats,
}

# Attach node positions if networkx ran successfully
try:
    result['vis_pair_idx']  = pair_idx
    result['vis_all_nodes'] = sorted(all_vis_nodes)
    result['vis_pos']       = {int(n): (float(xy[0]), float(xy[1]))
                                for n, xy in pos.items()}
except NameError:
    pass

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(result, f)

print(f'Saved → {SAVE_PATH}')
print(f'  PPR sets:  {len(CLASSIC_ALPHAS)} alphas × {len(pairs)} pairs')
print(f'  k-hop sets: {len(K_HOPS)} values × {len(pairs)} pairs')
print()
print('Suggested alpha values for Learnable PPR (classic restart convention):')
print('  teleport_values = [0.50, 0.33, 0.25]  # →  ~1-hop, 2-hop, 3-hop')
